# 05 · Streaming: Windowing & Triggers

> **Module 7 · Streaming · Lectures 7.1–7.4**

In this notebook you will:

- Tell event time apart from processing time, and re-stamp file events
- Compare fixed, sliding, and session windows on the same data
- See `AfterCount` and `AfterWatermark` differ in *when* they fire
- Watch ACCUMULATING emit `2, 4, 5` while DISCARDING emits `2, 2, 1`
- Trigger Beam's real `Unsafe trigger: AfterWatermark() may lose data` error
- Read the course streaming pipeline and see how it wires it all together

---

## 1. Setup

Everything runs on the **Direct Runner** with no GCP account and no bill. The
streaming pipeline in `project/streaming_pipeline.py` reads a bounded NDJSON
file, so the *same* windowing code runs locally and on Dataflow.

Two imports matter for this module:

- `apache_beam.transforms.window` — `FixedWindows`, `SlidingWindows`, `Sessions`
- `apache_beam.transforms.trigger` — `AfterCount`, `AfterWatermark`, `AccumulationMode`


In [ ]:
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = COURSE_ROOT.parent
PROJECT_DIR = COURSE_ROOT / "project"
TESTS_DIR = PROJECT_DIR / "tests"
for p in (str(PROJECT_DIR), str(TESTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions, TypeOptions
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.util import assert_that, equal_to
from apache_beam.transforms.core import TimestampedValue
from apache_beam.transforms.trigger import (
    AccumulationMode,
    AfterCount,
    AfterWatermark,
)
from apache_beam.transforms.window import FixedWindows, Sessions, SlidingWindows
from apache_beam.utils.timestamp import Timestamp

print(f"apache-beam {beam.__version__}")

# The course test options: the classic runner plus --allow_unsafe_triggers,
# which Beam requires before it will accept AfterCount.
from conftest import beam_test_options
from beam_options import resolve_runner

TEST_OPTS = beam_test_options()
print("test runner:", resolve_runner(TEST_OPTS))


def to_epoch(ts: str) -> float:
    """Convert an ISO-8601 timestamp to epoch seconds."""
    return datetime.fromisoformat(ts).timestamp()


def windowed_values(expected):
    """Build a matcher that compares the values, not the windows.

    With `use_global_window=False` the elements arrive as `WindowedValue`
    objects, so comparing them against plain tuples fails. This matcher
    unwraps the value and compares only the payload.

    Args:
        expected: The expected values, in any order.

    Returns:
        A Beam matcher.
    """

    def _compare(actual):
        unwrapped = [item.value if hasattr(item, "value") else item for item in actual]
        return sorted(unwrapped, key=repr) == sorted(expected, key=repr)

    return _compare


print("helpers ready")


## 2. Event time vs processing time

This is the single most important idea in streaming, and everything else in
this module hangs off it.

| Concept | What the clock is | Who controls it |
|---------|-------------------|-----------------|
| **Processing time** | When Beam *received* the element | The runner |
| **Event time** | When the event actually happened | The producer, in the payload |

A file source stamps every element with the time it was **read**, which is
processing time. If you window on processing time, every file lands in one
window and your "hourly" aggregation is a lie.

The fix is to **re-stamp** each element with the event's own timestamp, using
`TimestampedValue(value, timestamp)`.

Here are three events whose event times span two hours. Read them from a file
(no event timestamps), then re-stamp, and watch them fall into two different
windows.


In [ ]:
EVENT_LINES = [
    '{"event_id": "e1", "region": "EMEA", "event_ts": "2026-01-15T10:05:00+00:00", "value": 10.0}',
    '{"event_id": "e2", "region": "EMEA", "event_ts": "2026-01-15T10:55:00+00:00", "value": 20.0}',
    '{"event_id": "e3", "region": "EMEA", "event_ts": "2026-01-15T11:05:00+00:00", "value": 30.0}',
]


def parse_event(line):
    """Parse a JSON event line into (region, value, event_ts)."""
    payload = json.loads(line)
    return payload["region"], float(payload["value"]), payload["event_ts"]


def restamp(item):
    """Re-stamp an element with its own event time."""
    region, value, event_ts = item
    return TimestampedValue((region, value), to_epoch(event_ts))


# --- WITHOUT re-stamping: everything lands in one window ----------------
with TestPipeline(options=TEST_OPTS) as p:
    by_processing_time = (
        p
        | "ProcCreate" >> beam.Create(EVENT_LINES)
        | "ProcParse" >> beam.Map(parse_event)
        | "ProcKeyRegion" >> beam.Map(lambda item: (item[0], item[1]))
        | "ProcWindowFixed" >> beam.WindowInto(FixedWindows(3600))
        | "ProcKeyForCount" >> beam.Map(lambda kv: (kv[0], 1))
        | "ProcCount" >> beam.combiners.Count.PerKey()
    )
    assert_that(by_processing_time, equal_to([("EMEA", 3)]))

# --- WITH re-stamping: two windows, by event time -----------------------
with TestPipeline(options=TEST_OPTS) as p:
    by_event_time = (
        p
        | "EventCreate" >> beam.Create(EVENT_LINES)
        | "EventParse" >> beam.Map(parse_event)
        | "EventRestamp" >> beam.Map(restamp)
        | "EventWindowFixed" >> beam.WindowInto(FixedWindows(3600))
        | "EventKeyForCount" >> beam.Map(lambda pair: (pair[0], 1))
        | "EventCount" >> beam.combiners.Count.PerKey()
    )
    assert_that(by_event_time, windowed_values([("EMEA", 2), ("EMEA", 1)]), use_global_window=False)

print("processing time (no re-stamp): one window  -> [('EMEA', 3)]")
print("event time    (re-stamped)   : two windows -> [('EMEA', 2), ('EMEA', 1)]")
print()
print("The second run produced two groups of 2 and 1, because the three events")
print("straddle 10:00-11:00 and 11:00-12:00 in event time.")


## 3. Three kinds of window

All three answer the same question — *which elements belong together?* — but
they draw the groups differently.

| Window | Groups by | Use when |
|--------|-----------|----------|
| **Fixed** | Contiguous, equally-sized time slots | Regular reporting periods ("revenue per hour") |
| **Sliding** | Overlapping, equally-sized slots | Rolling metrics ("traffic in the last 10 minutes") |
| **Session** | Runs of activity separated by a gap | User sessions, click bursts, transactions per visit |

The same five events, one minute apart, produce:

- **Fixed** (1-hour windows): one group of 5
- **Sliding** (10-minute window, 5-minute period): several overlapping groups
- **Session** (10-minute gap): one group of 5 — no gap big enough to split it


In [ ]:
FIVE_EVENTS = [(f"2026-01-15T10:0{i}:00+00:00", 1.0) for i in range(5)]


def _stamped(events):
    """Attach a Beam timestamp to each (iso_ts, value) pair."""
    return [(Timestamp(to_epoch(ts)), value) for ts, value in events]


def _count_per_key(events, window, pipeline):
    """Window the events and count them per region."""
    return (
        pipeline
        | "Create" >> beam.Create(_stamped(events))
        | "Window" >> beam.WindowInto(window)
        | "KeyRegion" >> beam.Map(lambda pair: ("EMEA", pair[1]))
        | "KeyForCount" >> beam.Map(lambda kv: (kv[0], 1))
        | "Count" >> beam.combiners.Count.PerKey()
    )


# --- Fixed: one group of 5 ----------------------------------------------
with TestPipeline(options=TEST_OPTS) as p:
    fixed = _count_per_key(FIVE_EVENTS, FixedWindows(3600), p)
    assert_that(fixed, equal_to([("EMEA", 5)]))

# --- Sliding: 10-minute window, 5-minute period ------------------------
with TestPipeline(options=TEST_OPTS) as p:
    sliding = _count_per_key(FIVE_EVENTS, SlidingWindows(size=600, period=300), p)
    assert_that(
        sliding,
        windowed_values([("EMEA", 1), ("EMEA", 2), ("EMEA", 2), ("EMEA", 2), ("EMEA", 2), ("EMEA", 2), ("EMEA", 1)]),
        use_global_window=False,
    )

# --- Session: a 10-minute gap is never exceeded -------------------------
SESSION_EVENTS = [
    ("2026-01-15T10:05:00+00:00", 1.0),
    ("2026-01-15T10:06:00+00:00", 1.0),
    ("2026-01-15T10:20:00+00:00", 1.0),
]
with TestPipeline(options=TEST_OPTS) as p:
    sessions = _count_per_key(SESSION_EVENTS, Sessions(gap_size=600), p)
    assert_that(sessions, windowed_values([("EMEA", 2), ("EMEA", 1)]), use_global_window=False)

print("Fixed   (1h)                 : 1 group  -> [('EMEA', 5)]")
print("Sliding (10m window, 5m gap) : 7 groups -> 1, 2, 2, 2, 2, 2, 1")
print("Session (10m gap)            : 2 groups -> [('EMEA', 2), ('EMEA', 1)]")
print()
print("Sliding multiplies the work: every element joins several windows.")
print("Session is the only one that adapts to the data — its boundaries are")
print("decided by where the activity stops.")


## 4. Triggers: when does a window fire?

A **trigger** decides *when* a window's result is emitted. Two trigger types
matter here:

| Trigger | Fires when | Good for |
|---------|------------|----------|
| `AfterWatermark()` | The watermark passes the end of the window | Final, correct results |
| `AfterCount(n)` | `n` elements have arrived in the window | Early, incremental results |

The **watermark** is Beam's estimate of "no more data for time T will arrive".
It advances as data flows in, and it can lag reality — that lag is exactly
what **late data** is.

`AfterCount` is *unsafe*: a window can fire while more elements for it are
still in flight. Beam refuses to build such a pipeline unless you explicitly
opt in with `--allow_unsafe_triggers` — which is exactly what the course test
options do. The next cell asks Beam to build one anyway, so you can read the
real error:

```
ValueError: GroupByKey: Unsafe trigger: `AfterCount(2)` may lose data.
Reason: MAY_FINISH. This can be overriden with the --allow_unsafe_triggers flag.
```

`MAY_FINISH` is Beam's word for "this trigger can fire while elements are
still on their way". The optimisations that make a streaming job fast all
carry that risk, which is why Beam makes you opt in.

## 5. ACCUMULATING vs DISCARDING

This is the part that surprises people, so let's be precise.

- **ACCUMULATING** keeps everything the window has seen and re-emits the
  **running total** each time it fires.
- **DISCARDING** throws away the accumulated state after each firing and emits
  only the **delta** since the last firing.

Take five elements arriving one at a time, with `AfterCount(2)`:

| Firing | Elements since last firing | ACCUMULATING emits | DISCARDING emits |
|--------|---------------------------|--------------------|------------------|
| 1st | 2 | **2** | **2** |
| 2nd | 2 | **4** | **2** |
| 3rd | 1 | **5** | **1** |

So ACCUMULATING gives `2, 4, 5` and DISCARDING gives `2, 2, 1`. The
**final** value is the same (5) — but only if you sum the deltas. If your
downstream sink treats each emission as the current total (which is what a
BigQuery `MERGE` or an upsert does), ACCUMULATING is what you want.

This is exactly the distinction the course test suite pins down.


In [ ]:
# The course test options opt in to unsafe triggers; the default ones do not.
print("allow_unsafe_triggers:", TEST_OPTS.view_as(TypeOptions).allow_unsafe_triggers)

try:
    with beam.Pipeline(options=PipelineOptions(["--runner=BundleBasedDirectRunner"])) as p:
        (
            p
            | "Create" >> beam.Create([1, 2, 3])
            | "Window" >> beam.WindowInto(
                FixedWindows(60),
                trigger=AfterCount(2),
                accumulation_mode=AccumulationMode.DISCARDING,
            )
            | "KeyForCount" >> beam.Map(lambda value: ("k", 1))
            | "Count" >> beam.combiners.Count.PerKey()
        )
    print("NO ERROR")
except Exception as exc:
    print("ERROR TYPE:", type(exc).__name__)
    print("ERROR MSG :", exc)


In [ ]:
FIVE = [(f"2026-01-15T10:0{i}:00+00:00", 1.0) for i in range(5)]

with TestPipeline(options=TEST_OPTS) as p:
    accumulating = (
        p
        | "AccCreate" >> beam.Create(_stamped(FIVE))
        | "AccWindow" >> beam.WindowInto(
            FixedWindows(3600),
            trigger=AfterCount(2),
            accumulation_mode=AccumulationMode.ACCUMULATING,
        )
        | "AccKeyRegion" >> beam.Map(lambda pair: ("EMEA", pair[1]))
        | "AccKeyForCount" >> beam.Map(lambda kv: (kv[0], 1))
        | "AccCount" >> beam.combiners.Count.PerKey()
    )
    assert_that(
        accumulating,
        windowed_values([("EMEA", 2), ("EMEA", 4), ("EMEA", 5)]),
        use_global_window=False,
    )

with TestPipeline(options=TEST_OPTS) as p:
    discarding = (
        p
        | "DisCreate" >> beam.Create(_stamped(FIVE))
        | "DisWindow" >> beam.WindowInto(
            FixedWindows(3600),
            trigger=AfterCount(2),
            accumulation_mode=AccumulationMode.DISCARDING,
        )
        | "DisKeyRegion" >> beam.Map(lambda pair: ("EMEA", pair[1]))
        | "DisKeyForCount" >> beam.Map(lambda kv: (kv[0], 1))
        | "DisCount" >> beam.combiners.Count.PerKey()
    )
    assert_that(
        discarding,
        windowed_values([("EMEA", 2), ("EMEA", 2), ("EMEA", 1)]),
        use_global_window=False,
    )

print("ACCUMULATING -> [('EMEA', 2), ('EMEA', 4), ('EMEA', 5)]   (running totals)")
print("DISCARDING   -> [('EMEA', 2), ('EMEA', 2), ('EMEA', 1)]   (deltas)")


## 6. Allowed lateness, and the `AfterWatermark` safety error

Now the crucial safety rule, and the real error message.

`AfterWatermark()` fires when the watermark passes the window's end. That is
safe **only when allowed lateness is zero**. The moment you allow late data,
a window can be updated *after* it has already fired — and Beam will refuse,
because a trigger that can lose data is a trigger that can silently drop
records.

Let's ask Beam to do the unsafe thing and read the exact error.


In [ ]:
try:
    with beam.Pipeline(options=PipelineOptions(["--runner=BundleBasedDirectRunner"])) as p:
        (
            p
            | "Create" >> beam.Create([1, 2, 3])
            | "Window" >> beam.WindowInto(
                FixedWindows(3600),
                trigger=AfterWatermark(),
                accumulation_mode=AccumulationMode.ACCUMULATING,
                allowed_lateness=300,
            )
            | "KeyForCount" >> beam.Map(lambda value: ("EMEA", 1))
            | "Count" >> beam.combiners.Count.PerKey()
        )
    print("NO ERROR")
except Exception as exc:
    print("ERROR TYPE:", type(exc).__name__)
    print("ERROR MSG :", exc)
    print()

# And here is how the course pipeline handles the same situation.
import inspect

from streaming_pipeline import build_streaming_pipeline

source = inspect.getsource(build_streaming_pipeline)
start = source.index("unsafe = ")
end = source.index("window = FixedWindows")
print(source[start:end].rstrip())


There it is — the exact error the course pipeline is designed around:

```
ValueError: GroupByKey: Unsafe trigger: `AfterWatermark()` may lose data.
Reason: MAY_FINISH. This can be overriden with the --allow_unsafe_triggers flag.
```

Two things to take from it.

**Beam's safety model is precise.** `AfterWatermark` is safe when the window
is final, and only then. Allowed lateness means "this window is *not* final",
so the watermark trigger stops being safe. Beam knows this and says so.

**The override is a deliberate act.** `--allow_unsafe_triggers` silences the
error and logs a warning instead. In production that flag means *I have thought
about the trade-off and I accept that a window may be updated after it fired*.

The course streaming pipeline takes the safe path by default: when
`allowed_lateness` is non-zero and the flag is not set, it **falls back to
zero lateness** rather than crashing. Read that logic again:

- `trigger_count` is 0 → use `AfterWatermark` with `ACCUMULATING`.
- `trigger_count` is positive → use `AfterCount` with `DISCARDING`.
- Allowed lateness > 0 **and** no `--allow_unsafe_triggers` → log a warning and
  drop the lateness back to 0.

That last branch is the important one. It is better to run with *less* lateness
than to crash the job — and the warning tells the operator exactly what
happened.


## 7. The course streaming pipeline, end to end

Let's run it for real, over the sample events file, and inspect the output.
The pipeline lives in `project/streaming_pipeline.py` and uses the same
transforms we have been building by hand.

One quirk to know about: `build_streaming_pipeline` aggregates a *count* and a
*sum* separately and merges them with `CoGroupByKey`. There is no
`beam.combiners.Sum` in the Beam SDK — a sum is just the builtin `sum`
function, wrapped with `CombinePerKey`.


In [ ]:
from streaming_pipeline import build_streaming_pipeline
from transforms import parse_event_payload

EVENTS_NDJSON = COURSE_ROOT / "sample_data" / "events.ndjson"
print("events file:", EVENTS_NDJSON, "->", sum(1 for _ in EVENTS_NDJSON.open()) if EVENTS_NDJSON.exists() else "missing")

STREAM_OUT = COURSE_ROOT / "_out" / "nb05_stream"
RESTAMP_OUT = COURSE_ROOT / "_out" / "nb05_stream_restamped"
STREAM_OUT.parent.mkdir(parents=True, exist_ok=True)


def _restamp_payload(payload):
    """Re-stamp a raw event payload with the event's own timestamp."""
    event = parse_event_payload(payload)
    return TimestampedValue(payload, to_epoch(str(event["event_ts"])))


def _rows_for(path):
    """Read the data rows written by the streaming pipeline to one output path."""
    files = [f for f in path.parent.glob(path.name) if f.is_file()]
    text = "".join(f.read_text(encoding="utf-8") for f in files)
    return [line for line in text.splitlines()[1:] if line.strip()]


# --- Run 1: as-is (arrival timestamps from the file source) -------------
with beam.Pipeline(options=TEST_OPTS) as p:
    events = p | "ArrivalReadEvents" >> beam.io.ReadFromText(str(EVENTS_NDJSON))
    build_streaming_pipeline(
        p,
        events=events,
        output_path=str(STREAM_OUT),
        window_seconds=3600,
        allowed_lateness_seconds=0,
        trigger_count=0,
        options=TEST_OPTS,
    )

# --- Run 2: the same events, re-stamped with their event time -----------
with beam.Pipeline(options=TEST_OPTS) as p:
    events = p | "EventTimeReadEvents" >> beam.io.ReadFromText(str(EVENTS_NDJSON))
    restamped = events | "EventTimeRestamp" >> beam.Map(_restamp_payload)
    build_streaming_pipeline(
        p,
        events=restamped,
        output_path=str(RESTAMP_OUT),
        window_seconds=3600,
        allowed_lateness_seconds=0,
        trigger_count=0,
        options=TEST_OPTS,
    )

for label, path in (("arrival time ", STREAM_OUT), ("event time   ", RESTAMP_OUT)):
    rows = _rows_for(path)
    print(f"{label}: {len(rows):>4} (region, window) rows")
    for line in rows[:4]:
        print("      ", line)


Look closely at the arrival-time output: **four rows, one per region**.
With a one-hour window and a whole day of events, you would expect roughly 24
windows per region — not one.

That is not a bug in the course pipeline. It is the **arrival-time windowing
trap** from section 2, showing up in a real pipeline. `ReadFromText` stamps
every element with the same processing timestamp — the time the file was read —
so all 20,000 events land in the same one-hour window.

In production this pipeline reads from Pub/Sub, where the publisher's timestamp
*is* the event time, so the windows come out right. The file mode is a testing
convenience — and a live demonstration of why re-stamping matters.

The re-stamped run proves it: the same 20,000 events, the same window size,
the same pipeline code, but a completely different number of output rows,
because the events are now grouped by when they *happened* rather than when
they were *read*. That is the whole lesson of this module in one comparison.
Everything else — triggers, accumulation modes, allowed lateness — only makes
sense once the elements are in the right window to begin with.

One more detail about the output: each line is `region,event_count,total_value`
for one (region, window) pair, and the header comes from
`WriteToText(header=...)`. Notice what the output *is not*: it has no timestamp
column. The window is carried by Beam itself, not by the key. In production
you almost always want the window start in the output — otherwise you cannot
tell which hour a row belongs to. That means adding a `beam.DoFn.TimestampParam`
to read the window boundary, exactly as `ParseOrderFn` in
`project/transforms.py` already does for its own timestamp.

## 8. Reading a stream deterministically: `TestStream`

`TestStream` lets you construct a stream **in code** — elements at chosen
timestamps, the watermark advanced by hand — instead of waiting for real data.
That is how you test late data and re-triggering behaviour without a Pub/Sub
topic and without any sleeping.


In [ ]:
from apache_beam.testing.test_stream import TestStream

BASE = 1_700_000_000  # a fixed epoch so the test is deterministic

with TestPipeline(options=TEST_OPTS) as p:
    stream = (
        p
        | "TestStream" >> TestStream()
        .add_elements([Timestamp(BASE), Timestamp(BASE + 10), Timestamp(BASE + 20)])
        .advance_watermark_to(Timestamp(BASE + 60))
        .advance_watermark_to_infinity()
    )
    offsets = stream | "Offsets" >> beam.Map(lambda ts: int(ts.seconds()) - BASE)
    assert_that(offsets, equal_to([0, 10, 20]))

print("TestStream produced elements at offsets 0, 10 and 20 seconds.")
print("The watermark was advanced to +60 and then to infinity, which is what")
print("makes a test pipeline flush its final windows.")


## 9. What to change for production

| Concern | Notebook version | Production version |
|---------|------------------|--------------------|
| Event time | Re-stamped from the payload | Re-stamped from the payload, with the schema enforced |
| Trigger | `AfterWatermark`, zero lateness | Chosen per use case: `AfterProcessingTime` for dashboards, `AfterCount` for early results |
| Allowed lateness | Zero | A real value, with `--allow_unsafe_triggers` set deliberately |
| Late data | Dropped | Routed to a side output or a DLQ topic, with the window re-fired |
| Window in output | Not written | Window start written, so rows are joinable |
| Watermark lag | Not monitored | Watermark lag exported as a metric and alerted on |

The single most common production bug in this area is **arrival-time
windowing**: a pipeline that looks correct, has no late-data handling, and
quietly reports the wrong numbers because it grouped by when the data arrived
rather than when the event happened. Re-stamping is the fix, and it is two
lines of code.


## 🔑 Takeaways

- **Event time and processing time are different clocks.** A file source stamps processing time; re-stamp with `TimestampedValue(value, timestamp)` to window on when the event actually happened.
- **Fixed** windows give regular reporting slots, **sliding** windows give rolling metrics (and multiply the work), **session** windows adapt to bursts of activity.
- A **trigger** decides when a window fires: `AfterWatermark` waits for the window to be final, `AfterCount` fires early and is *unsafe*.
- **ACCUMULATING emits `2, 4, 5`** (running totals); **DISCARDING emits `2, 2, 1`** (deltas). The final value is the same only if you sum the deltas.
- **Allowed lateness breaks the `AfterWatermark` safety guarantee.** Beam's real error is `ValueError: GroupByKey: Unsafe trigger: `AfterWatermark()` may lose data. Reason: MAY_FINISH.` — and the override flag is a deliberate decision, not a convenience.
- **The course pipeline degrades safely**: with lateness and no `--allow_unsafe_triggers`, it logs a warning and falls back to zero lateness rather than crashing.
- **There is no `beam.combiners.Sum`** — a sum is `beam.CombinePerKey(sum)`, merged with counts using `CoGroupByKey`.
- **`TestStream`** makes late data and re-triggering testable deterministically, with no Pub/Sub topic and no sleeping.
